# E1.3 · Risk tiering agentic use cases

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.2 · Building the AI and agent inventory](https://spbreed.github.io/cyber-commons/lessons/E1.2.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Tier ten real workflows and assign approval authority.

**Why a security engineer needs it.** Tiering by model name instead of by what the thing can do. The control it builds is: autonomy level × action class × data sensitivity.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A single heavy control set applied to everything means the low-risk agents are over-governed, the high-risk ones are under-governed, and everybody routes around the process. Tiering is how proportionality becomes something you can write down.

> **At CyberTravels.** The RAG Advisor and the Workflow Agent do not deserve the same control set. One recommends hotels; the other moves money. R1, R12.

## 2 · The framework

```
   tier by three axes, not by product name

   autonomy      proposes -> acts with approval -> acts alone
   data          public -> internal -> regulated
   blast radius  read -> write -> irreversible

   tier 1  light control set    tier 3  the full set + verification
   one heavy default means everyone routes around the process
```

Risk-tier by what the system **can do**, not by which model it uses.

Tiering on model capability is the common mistake and it tracks vendor marketing
rather than exposure: every GPT-class deployment becomes "high" and every small
model "low". That gets the answer exactly backwards for the case that matters —
a small local model with production deploy rights and regulated data.

Three inputs determine consequence, and none of them is the model:

- **Autonomy** — what its output can trigger without a human.
- **Data** — what it can read, especially regulated or customer data.
- **Reach** — whether it can act externally.

The model matters for *likelihood* of a bad output, which is a different and
smaller term than consequence.

## 3 · The procedure, as a skill

The skill tiers five assets by autonomy, data and reach, then re-tiers them with the questionnaire that leads with the model question — and reports the inversion, where a small local model with deploy rights and regulated data moves from low to critical.

In [ ]:
# skills/grc/agentic-risk-tiering/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: agentic-risk-tiering
description: >-
  Tier an AI use case by what it can do — autonomy, data reach and external
  effect — and compare the result against tiering by which model it uses. Use
  when writing or auditing a risk questionnaire, or when every large-model
  deployment is coming out high.
allowed-tools: Read, Grep, Glob
---

# Tier the authority, not the model

Tiering on model capability tracks vendor marketing: every frontier deployment
becomes high and every small model low. It gets the important case backwards — a
small local model with production deploy rights and regulated data — because the
model determines the likelihood of a bad output and the authority determines
what a bad output costs.

## When to use this

Writing an intake questionnaire, auditing an existing one, or re-tiering a
portfolio whose distribution looks like the vendor's price list.

## Procedure

**1 — Score three inputs, none of which is the model.** What the output can
trigger without a human, what data it can read, and whether it can act
externally. Each on a small ordinal scale, and write the scale down.

**2 — Set thresholds and apply them.** Publish the thresholds with the tiers, so
a disputed tier is a dispute about a number rather than about judgement.

**3 — Tier the same portfolio by model, as a comparison.** Run the
questionnaire that leads with the model question and record where the two
disagree. The disagreements are the argument.

**4 — Look hardest at the inversions.** An asset that is low by model and
critical by authority is the case that motivates the change, and there is
usually one.

**5 — Write the four questions the intake form should ask** and, explicitly, the
one it should not lead with. Reviewers copy questionnaires; make yours the one
worth copying.

## Output contract

```json
{
  "scale": {"autonomy": ["str"], "data": ["str"], "reach": ["str"]},
  "assets": [{"name": "str", "autonomy": 0, "data": 0, "reach": 0, "score": 0, "tier": "low|medium|high|critical"}],
  "by_model": [{"name": "str", "tier": "low|medium|high|critical"}],
  "disagreements": [{"name": "str", "by_authority": "str", "by_model": "str", "inversion": true}],
  "questions": ["str"]
}
```

## Failure modes

- **Leading with the model question.** Everything downstream inherits it.
- **Unpublished thresholds.** The tier becomes an opinion.
- **Ignoring the inversions.** They are the whole finding.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/grc/agentic-risk-tiering/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/grc/agentic-risk-tiering/scripts/agentic_risk_tiering.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Tier a use case by autonomy, data and reach, and compare the answer against tiering by model.

This is the executable half of the `agentic-risk-tiering` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

from dataclasses import dataclass

@dataclass
class AIAsset:
    name: str; kind: str; owner: str = ""; autonomy: str = "L1"
    data: tuple = (); external: bool = False; registered: bool = True

TIER_THRESHOLDS = [(9, "critical"), (6, "high"), (3, "medium"), (0, "low")]

def risk_tier(a):
    score, why = 0, []
    pts = {"L1": 0, "L2": 1, "L2.5": 3, "L3": 5}[a.autonomy]
    if pts: score += pts; why.append(f"autonomy {a.autonomy} (+{pts})")
    if "regulated" in a.data: score += 3; why.append("regulated data (+3)")
    if "customer" in a.data:  score += 2; why.append("customer data (+2)")
    if a.external:            score += 2; why.append("can act externally (+2)")
    if not a.registered:      score += 1; why.append("unregistered (+1)")
    tier = next(t for th, t in TIER_THRESHOLDS if score >= th)
    return {"tier": tier, "score": score, "because": why}

ASSETS = [
 AIAsset("frontier chatbot, public docs, read-only", "copilot", "x", "L1", ("public",)),
 AIAsset("small local model with prod deploy rights", "agent", "x", "L3",
         ("customer", "regulated"), True),
 AIAsset("mid model, gated writes, internal only", "agent", "x", "L2", ("employee",)),
 AIAsset("frontier model summarising customer tickets", "copilot", "x", "L1",
         ("customer",)),
 AIAsset("unregistered remediation agent", "agent", "", "L2.5", ("customer",),
         True, registered=False),
]
print(f"{'asset':46s}{'tier':10s}{'score':>6}")
print("-" * 66)
for a in ASSETS:
    t = risk_tier(a)
    print(f"{a.name:46s}{t['tier']:10s}{t['score']:>6}")
    for w in t["because"]:
        print(f"{'':46s}{w}")

MODEL_TIER = {   # the questionnaire that asks 'which model?' first
 "frontier chatbot, public docs, read-only": "high",
 "small local model with prod deploy rights": "low",
 "mid model, gated writes, internal only": "medium",
 "frontier model summarising customer tickets": "high",
 "unregistered remediation agent": "medium",
}
print(f"{'asset':46s}{'by model':10s}{'by authority':14s}agreement")
print("-" * 84)
disagreements = 0
for a in ASSETS:
    by_auth = risk_tier(a)["tier"]
    by_model = MODEL_TIER[a.name]
    agree = by_auth == by_model
    disagreements += not agree
    print(f"{a.name:46s}{by_model:10s}{by_auth:14s}{'' if agree else '← DISAGREE'}")
print(f"\n{disagreements}/{len(ASSETS)} disagree.")
print("The worst inversion: the small local model with deploy rights tiers LOW")
print("on model capability and CRITICAL on what it can actually do.")
assert risk_tier(ASSETS[1])["tier"] == "critical"
assert MODEL_TIER[ASSETS[1].name] == "low"

QUESTIONS = [
 ("What can it change without a human approving that specific action?",
  "autonomy — the largest term"),
 ("What data can it read, and is any of it regulated or customer data?",
  "consequence of a leak"),
 ("Can it act outside our boundary?",
  "reach"),
 ("Is it registered, with a named owner?",
  "governability — an unowned asset cannot be remediated"),
]
NOT_ASKED = [
 "Which model does it use?",
 "How many parameters?",
 "Is the vendor SOC 2 certified?",
]
print("ASK:")
for q, why in QUESTIONS: print(f"   {q}\n      → {why}")
print("\nDO NOT tier on:")
for q in NOT_ASKED: print(f"   {q}")
print("   (these matter for LIKELIHOOD and vendor risk — a separate, smaller term)")

def tier_from_answers(can_change, reads_regulated, reads_customer, external, registered):
    a = AIAsset("x", "agent", "o" if registered else "", can_change,
                tuple(filter(None, ("regulated" if reads_regulated else "",
                                    "customer" if reads_customer else ""))),
                external, registered)
    return risk_tier(a)["tier"]

print("\nworked example — a new request:")
print("   'an agent that can issue refunds up to £500, reads customer orders,'")
print("   'runs internally, owned by payments-eng'")
print("   tier:", tier_from_answers("L2.5", False, True, False, True))

## What you just proved

The public read-only chatbot tiers low; the small local model with deploy rights and regulated data tiers critical at score 12. Tiering by model disagrees on 4 of 5 assets, most sharply inverting the small local model from low to critical. The worked example tiers the refund agent as high.

## Your turn

Re-tier your top ten AI use cases using only the four questions. Note which ones move, and be ready to explain the movement to whoever wrote the original questionnaire — the model question is usually question one.

---

**Next → [E1.4 · Control mapping for agents](https://spbreed.github.io/cyber-commons/lessons/E1.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*